In [1]:
import argparse
import numpy as np
import pandas as pd
import pvlib
import pytz
import laspy
import plotly.graph_objects as go
from visualize_rays_3d import *

# ============================================================================
# CONFIG
# ============================================================================

DEFAULT_LIDAR      = "output/recovered.laz"
DEFAULT_SHADOW_CSV = "results/shadow_matrix_results_SE_pro/shadow_attenuation_matrix_conecasting_SE_v10.csv"

LAT, LON       = 62.979849, 27.648656
LOCAL_TZ       = "Europe/Helsinki"

ARRAY_CORNER_XY    = np.array([532882.00, 6983507.00])
ROOF_SEARCH_RADIUS = 3.0
OFFSET_FROM_ROOF   = -0.5
TILT_DEG, AZ_DEG   = 12, 170
PANEL_W, PANEL_H   = 1.0, 1.6
ROW_CONFIG         = (5, 4, 3)

DEFAULT_RADIUS  = 50
RAY_LENGTH      = 80
POINT_SUBSAMPLE = 3

DEFAULT_VOXEL_SIZE         = 2.0
DEFAULT_VOXEL_OPACITY      = 0.0      # 0 = pure wireframe; bump for faint faces
DEFAULT_VOXEL_EDGE_WIDTH   = 2
DEFAULT_VOXEL_EDGE_OPACITY = 0.6

DEFAULT_RAY_CONE_ANGLE_DEG = 3.0      # 0 → single-line rays
DEFAULT_RAY_CONE_SEGMENTS  = 8
DEFAULT_RAY_COLOR          = "rgb(255,170,0)"
DEFAULT_RAY_OPACITY        = 0.30
# Edge length of the cube voxels representing occupied space. Adjust based on LiDAR density and desired visual clarity.


CLASS_COLORS = {
    2: ("Ground",   "rgb(180,150,120)"),
    3: ("Low Veg",  "rgb(144,210,144)"),
    4: ("Med Veg",  "rgb(60,160,60)"),
    5: ("High Veg", "rgb(20,100,20)"),
    6: ("Building", "rgb(200,60,60)"),
}

_CLASS_PRIORITY = {6: 5, 5: 4, 4: 3, 3: 2, 2: 1}


In [2]:
args = argparse.Namespace(
    time="2021-07-26 15:00",
    lidar=DEFAULT_LIDAR,
    shadow=DEFAULT_SHADOW_CSV,
    radius=DEFAULT_RADIUS,
    ray_length=RAY_LENGTH,
    subsample=POINT_SUBSAMPLE,
)

fig = visualize_scene(
    args.time,
    lidar_path=args.lidar,
    shadow_csv=args.shadow,
    radius=args.radius,
    ray_length=args.ray_length,
    subsample=args.subsample,
    # --- voxel display ---
    show_points=True,        # set False to hide the raw LiDAR cloud
    show_voxels=False,        # toggle voxel cube rendering
    voxel_size=DEFAULT_VOXEL_SIZE,
    voxel_opacity=DEFAULT_VOXEL_OPACITY,
    voxel_edges=True,                                # wireframe edges on top
    voxel_edge_width=DEFAULT_VOXEL_EDGE_WIDTH,
    voxel_edge_opacity=DEFAULT_VOXEL_EDGE_OPACITY,
    voxel_classes=None,      # e.g. [5, 6] for high-veg + buildings only
    show_sun=False,           # toggle sun marker display
    show_rays=False,          # toggle solar ray line display
    ray_cone_angle_deg=2.0,  # set >0 to visualize rays as cones instead of lines
    ray_cone_segments=DEFAULT_RAY_CONE_SEGMENTS,
    ray_color=DEFAULT_RAY_COLOR,
    ray_opacity=DEFAULT_RAY_OPACITY,
)


Visualizing scene at 2021-07-26 15:00:00 (local)
  UTC: 2021-07-26 12:00:00+00:00
  Solar altitude: 42.9°
  Solar azimuth:  214.4°
  Loading LiDAR from output/recovered.laz...
  Array center: (532884.2, 6983509.2, 97.6)
  12 panel points
  Showing 24,295 points (of 72,884 within 50m)
  Matrix transmittance at (42.9°, 214.4°): 0.993
  Shadow factor: 0.007

  Saved interactive 3D view to scene_3d_20210726_1500.html
